# Biotic Hardware Synthesis — Comparative Morphology Demo v1.2.1

Full pipeline: parameter derivation, Schumann baseline, five morphologies (fractal, botanical, random control, Fibonacci spiral, Voronoi control), 3-metric statistics over 10 pairs, multi-seed distributions.

**Run from the repo root** (`biotic-hardware/`), not from inside `notebook/`. Generated artifacts are written to `outputs/`.

## 1 — Run the full pipeline

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, 'run.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 2 — Schumann resonance external baseline

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from data.schumann_reference import SCHUMANN_MODES_HZ, SCHUMANN_SOURCE, nearest_schumann_mode

with open('outputs/resonance_params.json') as f:
    res = json.load(f)

f_sim = res['f_resonance_Hz']
nearest_hz, mode_n, deviation = nearest_schumann_mode(f_sim)

fig, ax = plt.subplots(figsize=(10, 3))
for i, f in enumerate(SCHUMANN_MODES_HZ):
    color = '#FF5722' if i + 1 == mode_n else '#BDBDBD'
    label = f'Mode {i+1} ({f} Hz)' if i + 1 == mode_n else f'Mode {i+1}'
    ax.axvline(f, color=color, linewidth=2, label=label)
ax.axvline(f_sim, color='#2196F3', linewidth=2.5, linestyle='--',
           label=f'Simulated ({f_sim:.4f} Hz)')
ax.set_xlabel('Frequency (Hz)')
ax.set_title(
    f'Schumann Resonance Comparison - deviation {deviation:.2f}% from mode {mode_n}'
    f' | {SCHUMANN_SOURCE}'
)
ax.set_yticks([])
ax.legend(fontsize=9)
ax.set_xlim(0, 40)
plt.tight_layout()
plt.show()

print(f'Simulated : {f_sim:.4f} Hz')
print(f'Nearest   : {nearest_hz:.2f} Hz  (mode {mode_n})')
print(f'Deviation : {deviation:.2f}%')

## 3 — Morphological sensitivity curves (normalized)

Five morphologies. Solid = Merit Scaled, dashed = Coherence Ratio.

In [ ]:
import pandas as pd

paths = {
    'Fractal':   'outputs/simulation_results_fractal.csv',
    'Botanical': 'outputs/simulation_results_botanical.csv',
    'Random':    'outputs/simulation_results_random.csv',
    'Fibonacci': 'outputs/simulation_results_fibonacci.csv',
    'Voronoi':   'outputs/simulation_results_voronoi.csv',
}
colors = {
    'Fractal':   '#2196F3',
    'Botanical': '#4CAF50',
    'Random':    '#FF5722',
    'Fibonacci': '#9C27B0',
    'Voronoi':   '#FF9800',
}

def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

fig, ax = plt.subplots(figsize=(14, 6))
for name, path in paths.items():
    df = pd.read_csv(path)
    ax.plot(df['Distance'], norm(df['Merit_Scaled']), label=f'{name} - Merit Scaled',
            linewidth=2, color=colors[name])
    ax.plot(df['Distance'], norm(df['Coherence_Ratio']), label=f'{name} - Coherence',
            linewidth=1.1, color=colors[name], linestyle='--', alpha=0.55)

ax.set_xlabel('Distance (m)')
ax.set_ylabel('Normalized value')
ax.set_title('Morphological Sensitivity Benchmark v1.2.1 - Five Morphologies (seed 42)')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 4 — Statistical separation: 3 metrics x 10 pairs

Welch t-test + Cohen's d read from `outputs/statistical_summary.csv` (generated by the pipeline).  
**p-value**: green = significant. **|Cohen's d|**: blue = large effect.

In [ ]:
df_stat = pd.read_csv('outputs/statistical_summary.csv')

metrics = ['Merit_Scaled', 'Coherence_Ratio', 'Peak_AF']
pairs   = [
    'Fractal vs Botanical', 'Fractal vs Random', 'Fractal vs Fibonacci',
    'Fractal vs Voronoi', 'Botanical vs Random', 'Botanical vs Fibonacci',
    'Botanical vs Voronoi', 'Random vs Fibonacci', 'Random vs Voronoi',
    'Fibonacci vs Voronoi',
]

p_matrix = np.zeros((len(metrics), len(pairs)))
d_matrix = np.zeros((len(metrics), len(pairs)))

for _, row in df_stat.iterrows():
    i = metrics.index(row['Metric'])
    j = pairs.index(row['Pair'])
    p_matrix[i, j] = row['p_value']
    d_matrix[i, j] = abs(row['Cohens_d'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8))

im1 = ax1.imshow(p_matrix, cmap='RdYlGn_r', vmin=0, vmax=0.1, aspect='auto')
ax1.set_xticks(range(len(pairs)))
ax1.set_xticklabels(pairs, rotation=35, ha='right', fontsize=8)
ax1.set_yticks(range(len(metrics)))
ax1.set_yticklabels(metrics)
ax1.set_title('p-value (green = significant, threshold 0.05)')
for i in range(len(metrics)):
    for j in range(len(pairs)):
        txt_color = 'white' if p_matrix[i, j] < 0.01 else 'black'
        ax1.text(j, i, f'{p_matrix[i,j]:.3f}', ha='center', va='center',
                 fontsize=8, color=txt_color)
plt.colorbar(im1, ax=ax1)

d_max = max(d_matrix.max(), 1.0)
im2 = ax2.imshow(d_matrix, cmap='Blues', vmin=0, vmax=d_max, aspect='auto')
ax2.set_xticks(range(len(pairs)))
ax2.set_xticklabels(pairs, rotation=35, ha='right', fontsize=8)
ax2.set_yticks(range(len(metrics)))
ax2.set_yticklabels(metrics)
ax2.set_title("|Cohen's d| (dark blue = large effect >0.8)")
for i in range(len(metrics)):
    for j in range(len(pairs)):
        txt_color = 'white' if d_matrix[i, j] > d_max * 0.6 else 'black'
        ax2.text(j, i, f'{d_matrix[i,j]:.2f}', ha='center', va='center',
                 fontsize=8, color=txt_color)
plt.colorbar(im2, ax=ax2)

fig.suptitle('Statistical Separation - Welch t-test + Cohen d (v1.2.1)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

df_stat[['Metric', 'Pair', 'p_value', 'Significant_p05', 'Cohens_d', 'Effect_size']]

## 5 — Multi-seed analysis: distributions by morphology (seeds 42–46)

Each bar is the mean over 5 seeds. Error bars = ±1 std.  
Fractal and Fibonacci std low → seed-stable. Botanical, Random and Voronoi std higher → seed-sensitive.

In [ ]:
df_multi = pd.read_csv('outputs/multi_seed_summary.csv')

morphologies = df_multi['Morphology'].tolist()
colors = {
    'fractal':   '#2196F3',
    'botanical': '#4CAF50',
    'random':    '#FF5722',
    'fibonacci': '#9C27B0',
    'voronoi':   '#FF9800',
}

metrics_multi = [
    ('Merit_Mean',     'Merit_Std',     'Merit Scaled'),
    ('Coherence_Mean', 'Coherence_Std', 'Coherence Ratio'),
    ('PeakAF_Mean',    'PeakAF_Std',    'Peak AF'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (mean_col, std_col, label) in zip(axes, metrics_multi):
    means = df_multi[mean_col].values
    stds  = df_multi[std_col].values
    bar_colors = [colors.get(m, '#888888') for m in morphologies]
    bars = ax.bar(morphologies, means, yerr=stds, color=bar_colors,
                  capsize=6, alpha=0.85, width=0.6)
    for bar, mean, std in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            mean + std + max(means) * 0.03,
            f'{mean:.4f}\n+/-{std:.4f}',
            ha='center', va='bottom', fontsize=7
        )
    ax.set_title(label)
    ax.set_ylabel('Value')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Multi-seed Sensitivity Analysis - seeds [42, 43, 44, 45, 46] (v1.2.1)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 — exploration_summary.json

Machine-readable output: parameter derivation + experimental configuration + Schumann baseline + multi-seed results in a single structured entry point.

In [ ]:
with open('outputs/exploration_summary.json') as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))